<a href="https://colab.research.google.com/github/kalleknast/VocDenoiser/blob/main/notebooks/colab_driver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VocDenoiser — Colab driver

Thin driver that calls into the `vocdenoiser` package. Three stages:
1. **SNR filtering** — score clips, pick a clean training subset (CPU, numpy-only).
2. **Train** the β-VAE denoiser (GPU).
3. **Architecture search** — autoresearch-style loop over model families (GPU).

Runtime → Change runtime type → **GPU (A100/L4)** for stages 2–3.

In [1]:
# Install (SNR-only: `pip install -e .`; with ML stack: `.[ml]`)
!git clone https://github.com/kalleknast/VocDenoiser.git
%cd VocDenoiser
!pip install -q -e '.[ml]'

Cloning into 'VocDenoiser'...
remote: Enumerating objects: 310, done.
remote: Counting objects: 100% (310/310), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 310 (delta 160), reused 255 (delta 108), pack-reused 0 (from 0)
Receiving objects: 100% (310/310), 156.12 KiB | 795.00 KiB/s, done.
Resolving deltas: 100% (160/160), done.
/content/VocDenoiser
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 62.9 MB/s eta 0:00:00
  Building editable for vocdenoiser (pyproject.toml) ... done


In [ ]:
# Get the clean calls AND the real colony-noise onto Colab-LOCAL disk (never train off Drive).
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/MarmosetData'   # where the three zips were uploaded
for z in ['Vocalizations', 'Noise', 'Cigarra']:
    print(f'Unzipping {z}.zip ...')
    !unzip -q -o '{DRIVE_DIR}/{z}.zip' -d /content
# Verify all three extracted (expect ~50000 / 30 / 136):
!echo "Vocalizations:" $(ls /content/Vocalizations 2>/dev/null | wc -l) \
      "| Noise:" $(ls /content/Noise 2>/dev/null | wc -l) \
      "| Cigarra:" $(ls /content/Cigarra 2>/dev/null | wc -l)
os.environ['VOCDENOISER_DATA_ROOT'] = '/content/Vocalizations'   # 50k clean phee calls
# Real recorded backgrounds for the training noise mix: /content/Noise, /content/Cigarra

# Durable OUTPUTS -> Drive, so they survive a runtime reset / disconnect and training + search
# can RESUME. Every command honors $VOCDENOISER_OUTPUT_ROOT; leave it unset (or run locally) and
# outputs fall back to repo-relative paths. INPUT audio stays on fast local /content (above);
# only the small, expensive-to-recompute outputs (checkpoints, ledger, SNR scan) go to Drive.
os.environ['VOCDENOISER_OUTPUT_ROOT'] = '/content/drive/MyDrive/VocDenoiser'
os.makedirs(os.environ['VOCDENOISER_OUTPUT_ROOT'] + '/artifacts', exist_ok=True)
print('outputs ->', os.environ['VOCDENOISER_OUTPUT_ROOT'])

Mounted at /content/drive
Unzipping Vocalizations.zip ...


## 1. SNR filtering — select a clean training subset

In [ ]:
# Score every clip, then inspect the distribution before choosing a cutoff.
# The scores CSV goes to Drive ($VOCDENOISER_OUTPUT_ROOT) — the 50k-clip scan is slow, so
# persisting it means a runtime reset doesn't force a re-scan; select (below) reads it back.
!python -m vocdenoiser.cli snr scan $VOCDENOISER_DATA_ROOT \
    --out "$VOCDENOISER_OUTPUT_ROOT/artifacts/snr_scores.csv" --workers 8
!python -m vocdenoiser.cli snr report "$VOCDENOISER_OUTPUT_ROOT/artifacts/snr_scores.csv" --out-dir reports
# (optional) confirm the SNR metric is call-agnostic on THIS data, using the real colony noise.
!python -m vocdenoiser.cli snr validate "$VOCDENOISER_OUTPUT_ROOT/artifacts/snr_scores.csv" \
    --src-dir $VOCDENOISER_DATA_ROOT --noise-dir /content/Noise --noise-dir /content/Cigarra --out-dir reports
from IPython.display import Markdown
Markdown(open('reports/snr_report.md').read())

In [ ]:
# Pick a threshold from the report, then select the clean subset.
# Decided recipe for this data: primary snr_db cutoff at the Otsu crossover
# (~19.82 dB on the current scan -> ~19.5k clips), plus a secondary broadband
# floor. The broadband floor is a cheap safeguard against hissy narrowband/tonal
# calls that the primary snr_db under-penalizes; at the 19.82 dB cutoff it only
# trims ~0.1% (the primary cutoff already excludes most of them), but it matters
# if you lower --threshold to recover more clips. Re-read reports/ for THIS scan.
# Reads the Drive scores CSV; the manifest + symlink subset are local (regenerated per session).
!python -m vocdenoiser.cli snr select "$VOCDENOISER_OUTPUT_ROOT/artifacts/snr_scores.csv" \
    --src-dir $VOCDENOISER_DATA_ROOT --out artifacts/clean_manifest.csv \
    --threshold 19.82 --broadband-floor 15.0 --link-dir /content/clean_subset

## 2. Train the β-VAE denoiser

In [ ]:
# Train the β-VAE denoiser at 44.1 kHz (config default — phee energy is <~22 kHz, so 96 kHz is
# cost without signal). Source WAVs are 96 kHz; resampling every item every epoch is CPU-bound
# and becomes the bottleneck on a low-core Colab VM (GPU starves waiting for batches). So
# pre-downsample the clean subset ONCE to a LOCAL 44.1 kHz copy and train off that. Idempotent:
# skips the downsample if the copy is already complete (re-run / resume just reuses it).
import glob
SRC_ROOT = '/content/clean_subset'         # SNR-selected symlinks (96 kHz) from section 1
TRAIN_ROOT = '/content/clean_subset_44k'    # local 44.1 kHz copy we actually train from
N_CPU = os.cpu_count() or 2                  # augmentation is CPU-bound; match workers to cores
n_src = len(glob.glob(SRC_ROOT + '/**/*.wav', recursive=True))
n_dst = len(glob.glob(TRAIN_ROOT + '/**/*.wav', recursive=True))
if n_dst < n_src:
    print(f'Pre-downsampling {n_src} clips to 44.1 kHz ({n_dst} already present)...')
    !python -m vocdenoiser.datasets.resample --src {SRC_ROOT} --dst {TRAIN_ROOT} \
        --target-sr 44100 --workers {N_CPU}
else:
    print(f'44.1 kHz training set ready: {n_dst} clips at {TRAIN_ROOT}')

# Checkpoints go to Drive ($VOCDENOISER_OUTPUT_ROOT/checkpoints) so a runtime reset/disconnect
# doesn't lose progress. RESUME is the default: re-running this cell picks up Drive's last.ckpt
# (full optimizer + epoch + early-stop state) via --resume-from auto and continues to --max-epochs.
#
# ⚠ RECIPE CHANGE: the production model now defaults to L1 reconstruction + KL warmup + AdamW
# (the architecture search's credible wins, promoted into the hand model). This changes the
# OBJECTIVE and the val_loss SCALE — L1·n_bins is a larger number than the old MSE·n_bins, so a
# higher val_loss here is NOT a regression, just a different unit. Checkpoints from an older
# MSE/Adam run are therefore incompatible: for the FIRST run on the new recipe, set
# FRESH_START = True (trains from scratch). Set it back to False afterwards to resume normally.
CKPT_DIR = os.environ['VOCDENOISER_OUTPUT_ROOT'] + '/checkpoints'
FRESH_START = True   # ← set True ONCE to adopt the new L1+warmup+AdamW recipe, then back to False
if FRESH_START:
    !rm -rf "{CKPT_DIR}"
    print('cleared', CKPT_DIR)

# Noise mix = synthetic recipe (40% pink / 50% babble / 10% transients) BLENDED with the real
# recorded backgrounds (--noise-dirs); tune with --real-noise-weight (0=synthetic, 1=real, def 0.5).
# --num-workers = CPU count: augmentation is CPU-bound, and over-subscribing cores thrashes.
# The recipe flags below are the new DEFAULTS, passed explicitly so the recipe is visible / easy
# to A/B (e.g. drop back to the old baseline with --recon-loss l2 --beta-schedule const --optimizer adam).
!python -m vocdenoiser.denoise.train \
    --data-root {TRAIN_ROOT} --max-epochs 100 --batch-size 32 --beta 4.0 \
    --recon-loss l1 --beta-schedule warmup --beta-warmup-epochs 5 \
    --optimizer adamw --weight-decay 1e-4 \
    --noise-dirs /content/Noise /content/Cigarra --num-workers {N_CPU} \
    --resume-from auto

In [ ]:
# Evaluate: UMAP of the 16-dim latents (+ optional RandomForest identity proxy).
# data/Vocalizations filenames are bare numeric call IDs with NO identity, so the UMAP is
# unlabelled and the RF proxy is skipped here; the real caller-identity number is section 2b.
# best_ckpt() picks the lowest-val_loss FINITE checkpoint from the Drive checkpoints dir
# (skips a NaN/poisoned last.ckpt).
import glob, re
CKPT_DIR = os.environ['VOCDENOISER_OUTPUT_ROOT'] + '/checkpoints'
def best_ckpt(d=CKPT_DIR):
    cands = [f for f in glob.glob(f"{d}/betavae-*val_loss=*.ckpt") if "nan" not in f.lower()]
    # Match the number precisely (digits.digits) — a greedy [0-9.]+ would swallow
    # the '.' before '.ckpt' and yield e.g. '276.1577.', which float() rejects.
    return min(cands, key=lambda f: float(re.search(r"val_loss=([0-9]+\.[0-9]+)", f).group(1))) \
        if cands else f"{d}/last.ckpt"
CKPT = best_ckpt(); print("checkpoint:", CKPT)

!python -m vocdenoiser.denoise.eval --data-root /content/clean_subset --ckpt "{CKPT}"

## 2b. Cross-dataset caller-identity benchmark (InfantMarmosetsVox)

An external check of "does compression preserve individual identity?" on a dataset with
ground-truth **caller identity** ([InfantMarmosetsVox](https://zenodo.org/records/10130104),
CC-BY-4.0; 10 infant marmosets, 11 call types). The loader downloads the audio, cuts each
annotated vocalization at **44.1 kHz — now the model's native rate, so no upsampling and no
empty high-frequency band** — **scores call quality** (SNR + clipping + level) and drops
noisy / near-silent clips, then writes an `id,identity` (=caller) CSV.

The raw audio is ~21 GB and Zenodo throttles, so after the first run the small **prepared clips
+ labels** are cached to `MyDrive/MarmosetData/InfantMarmosetsVox/` (clips as one tarball to
dodge the Drive-FUSE many-small-files penalty); re-runs restore from Drive and skip the download.

Read the RandomForest accuracy as a **cross-dataset** number — a different colony (infant
twins) — not comparable to an in-domain split. Needs the checkpoint trained in section 2.

In [ ]:
# Cross-dataset caller-identity benchmark. Needs the checkpoint from section 2 (uses CKPT /
# best_ckpt() from the eval cell above). Runs the eval on Colab-LOCAL clips (fast).
#
# The raw IMV audio is ~21 GB and Zenodo throttles downloads, so we CACHE the small PREPARED
# outputs — the cut clips (~a couple GB) + the labels CSV — to Drive under MarmosetData/. First
# run downloads + cuts, then caches; any later run (or a fresh runtime) restores from Drive and
# skips the download entirely. Clips are cached as ONE tarball, not thousands of loose WAVs, to
# avoid the Drive-FUSE many-small-files penalty. (DRIVE_DIR comes from section 0's Drive setup.)
import os
IMV_ROOT  = '/content/InfantMarmosetsVox'                # local working dir (ephemeral)
IMV_CACHE = DRIVE_DIR + '/InfantMarmosetsVox'            # persistent, under MarmosetData/ on Drive
CACHE_TAR = IMV_CACHE + '/imv_clips_44k.tar.gz'          # prepared clips, as a single file
CACHE_CSV = IMV_CACHE + '/imv_labels.csv'
os.makedirs(IMV_ROOT, exist_ok=True)

if os.path.exists(CACHE_TAR) and os.path.exists(CACHE_CSV):
    print('Restoring prepared IMV clips from the Drive cache — skipping the ~21 GB download.')
    !tar xzf "{CACHE_TAR}" -C {IMV_ROOT}                 # -> {IMV_ROOT}/clips
    !cp "{CACHE_CSV}" "{IMV_ROOT}/imv_labels.csv"
else:
    # Zenodo throttles per connection; aria2c pulls the tarballs over 16 connections (5-20x faster).
    !apt-get -qq install -y aria2
    # Download (~21 GB) + extract + cut per-call clips at 44.1 kHz (IMV native = model rate, a
    # no-op resample) + SCORE CALL QUALITY, dropping noisy / near-silent clips. Re-runs skip
    # already-extracted twins, so an interrupted download resumes.
    !python -m vocdenoiser.datasets.infantmarmosetsvox \
        --download --imv-root $IMV_ROOT --target-sr 44100 \
        --min-snr 6 --min-peak-dbfs -40
    # Cache the small prepared outputs to Drive so future runs never re-download. One tarball =
    # a single large write, fast on Drive FUSE. The raw 21 GB ($IMV_ROOT/data) is NOT cached.
    print('Caching prepared clips + labels to', IMV_CACHE)
    os.makedirs(IMV_CACHE, exist_ok=True)
    !tar czf "{CACHE_TAR}" -C {IMV_ROOT} clips
    !cp "{IMV_ROOT}/imv_labels.csv" "{CACHE_CSV}"

# (optional) reclaim ~21 GB of local disk once clips are cut/cached — data/ isn't needed after:
# !rm -rf $IMV_ROOT/data

# Identity-preservation proxy on the compressed latents of a DIFFERENT colony.
!python -m vocdenoiser.denoise.eval \
    --data-root $IMV_ROOT/clips --ckpt "{CKPT}" \
    --labels-csv $IMV_ROOT/imv_labels.csv \
    --out-png umap_imv.png --out-latents latents_imv.npy

from IPython.display import Image
Image('umap_imv.png')

## 3. Architecture search (autoresearch-style)

Greedy hill-climb over a top-K frontier under a fixed compute budget; each
candidate trained for `--max-steps` and scored by spectrogram SI-SDR. The JSONL
ledger is the resumable state.

In [ ]:
# Dry-run the loop with NO GPU (synthetic landscape) to see the mechanics:
!python -m vocdenoiser.cli search run --harness mock --iters 30 --ledger artifacts/mock.jsonl
!python -m vocdenoiser.cli search report --ledger artifacts/mock.jsonl

In [ ]:
# Real search (GPU): trains each candidate under a fixed budget on the clean subset.
# RE-RUNNING this cell RESUMES the search (skips already-evaluated candidates, keeps hill-
# climbing from the persisted frontier). Accumulate across sessions by just re-running.
#
# Tuning below is from analysing the 70-record ledger, which had COLLAPSED onto tiny
# 0.28M / latent-8 models (params<->metric correlation -0.54 = a small-budget artifact):
#   --max-steps 3000   2x the per-candidate budget (1500 still left mid/large models
#                      undertrained, so the metric rewarded fast-converging tiny nets over
#                      better architectures). Step-based = reproducible + comparable across
#                      Colab's mixed A100/L4. (A wall-clock --max-steps -1 --max-time
#                      DD:HH:MM:SS budget is fairer across model sizes but is NOT comparable
#                      between GPU tiers -- only use it if you pin one GPU for the campaign.)
#   FRESH LEDGER (search_ledger_v2.jsonl): the SI-SDR scale shifts with the budget, so new
#                      runs must NOT mix with the old 1500-step records -- the accept rule
#                      would compare incomparable metrics. (Old ledger left intact for ref.)
#   --max-params 4M    enforced on the COMPUTED param count. Bounding the choice sets
#                      (n_conv_layers<=4, base_channels<=48) does NOT bound size: params
#                      are dominated by the dense bottleneck over the flattened encoder
#                      output, which shrinks 4x per conv layer -- so the SHALLOWEST net is
#                      the biggest. 20% of the space still exceeded 4M and a 7.6M model
#                      reached the v2 ledger. Pass --max-params 0 to disable the cap.
#   --explore-rate 0.35  raised from 0.2 after search_ledger_v2: the incumbent came from a
#                      random restart, then 7 straight evolved children landed inside the
#                      noise band -- exploration was carrying the search, not the hill-climb.
#                      (Observed random share runs ~0.4: proposals that duplicate a known id
#                      are skipped, so only fresh ones reach the ledger.)
#   --seeds 0 1 2  average 3 seeds; --final-seeds 5
#                      re-evaluate the winner at 5 seeds (print-only). Per-candidate cost ~=
#                      seeds x per-seed train time; trim --iters / --seeds if it runs too long.
# NOTE: the metric is reconstruction SI-SDR ONLY -- it does NOT see caller-identity retention.
# Choose latent_dim from the downstream RF/UMAP eval (section 2b), not from this ledger alone.
# !rm -f "$VOCDENOISER_OUTPUT_ROOT/artifacts/search_ledger_v2.jsonl"   # uncomment to restart
!python -m vocdenoiser.cli search run --harness torch \
    --ledger "$VOCDENOISER_OUTPUT_ROOT/artifacts/search_ledger_v2.jsonl" \
    --data-root /content/clean_subset --iters 60 --max-steps 3000 \
    --seeds 0 1 2 --explore-rate 0.35 --final-seeds 5 \
    --noise-dirs /content/Noise /content/Cigarra
!python -m vocdenoiser.cli search report \
    --ledger "$VOCDENOISER_OUTPUT_ROOT/artifacts/search_ledger_v2.jsonl"

## 3b. Resume the search after a runtime reset

One self-contained cell: rebuilds the local `/content` inputs (wiped on a reset) and continues the search from the ledger persisted on Drive. Every setup step is guarded, so it's also safe to just re-run in a live session to add more iterations. Assumes a prior full run left `snr_scores.csv` and the search ledger under `$VOCDENOISER_OUTPUT_ROOT` on Drive.

In [ ]:
# === RESUME the architecture search ===============================================
# Rebuilds the local /content inputs the search reads (gone after a runtime reset) and
# picks up the Drive ledger where it left off. Each step is guarded -> fast no-op if the
# runtime is still warm, so re-running this cell simply continues the search.
import os, glob

# 1. Repo + package. Clone on a fresh runtime, PULL on a warm one.
#    'Restart runtime' keeps /content, so the clone below is skipped and the code would
#    otherwise stay pinned at whatever revision this runtime first cloned -- silently, since
#    nothing errors. That is not hypothetical: v2 trials 17-19 ran the pre-cap proposer
#    after a restart and re-tested 4.56M-param candidates the cap exists to reject.
#    The search runs as a fresh `!python -m ...` process against an editable install, so a
#    pull is enough -- the next candidate picks up the new code without a reinstall.
if not os.path.isdir('/content/VocDenoiser'):
    !git clone -q https://github.com/kalleknast/VocDenoiser.git
os.chdir('/content/VocDenoiser')
!git pull -q
try:
    import vocdenoiser  # noqa: F401
except ImportError:
    !pip install -q -e '.[ml]'
# The revision actually about to run. Check this against what you pushed BEFORE trusting a
# resumed run -- it is the cheap guard against analysing a ledger written by stale code.
!git log --oneline -1

# 2. Drive + audio + env vars. mount() no-ops if already mounted; unzip only if missing.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/MarmosetData'
for z in ['Vocalizations', 'Noise', 'Cigarra']:
    if not glob.glob(f'/content/{z}/*'):
        print(f'Unzipping {z}.zip ...')
        !unzip -q -o '{DRIVE_DIR}/{z}.zip' -d /content
os.environ['VOCDENOISER_DATA_ROOT']   = '/content/Vocalizations'
os.environ['VOCDENOISER_OUTPUT_ROOT'] = '/content/drive/MyDrive/VocDenoiser'
os.makedirs(os.environ['VOCDENOISER_OUTPUT_ROOT'] + '/artifacts', exist_ok=True)

# 3. Clean-subset symlinks that `search run --data-root` points at. Fast: reads the SNR
#    scores CSV already on Drive (no 50k re-scan). Same recipe as cell 5.
if not os.path.isdir('/content/clean_subset'):
    !python -m vocdenoiser.cli snr select "$VOCDENOISER_OUTPUT_ROOT/artifacts/snr_scores.csv" \
        --src-dir $VOCDENOISER_DATA_ROOT --out artifacts/clean_manifest.csv \
        --threshold 19.82 --broadband-floor 15.0 --link-dir /content/clean_subset

# 4. Show what we're resuming from (the ledger on Drive survives resets). v2 = the capped-
#    space / raised-budget regime; the old search_ledger.jsonl (1500-step) is a different
#    metric scale and is left untouched.
LEDGER = os.environ['VOCDENOISER_OUTPUT_ROOT'] + '/artifacts/search_ledger_v2.jsonl'
n = sum(1 for _ in open(LEDGER)) if os.path.exists(LEDGER) else 0
print(f'Resuming search from {n} existing records -> {LEDGER}')

# 5. Continue the search (skips already-evaluated candidates; keeps hill-climbing). Keep
#    these flags in sync with the standard search cell above (v2 ledger, --max-steps 3000).
!python -m vocdenoiser.cli search run --harness torch \
    --ledger "$VOCDENOISER_OUTPUT_ROOT/artifacts/search_ledger_v2.jsonl" \
    --data-root /content/clean_subset --iters 60 --max-steps 3000 \
    --seeds 0 1 2 --explore-rate 0.35 --final-seeds 5 \
    --noise-dirs /content/Noise /content/Cigarra
!python -m vocdenoiser.cli search report \
    --ledger "$VOCDENOISER_OUTPUT_ROOT/artifacts/search_ledger_v2.jsonl"